# Anima 샘플러 비교 · 임시 실험

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_diffsynth_practive/blob/feature/anima-samplers/tmp_anima_sampler_compare_colab.ipynb?forceEdit=true&sandboxMode=true)

기존 생성 프롬프트·설정을 불러와 **Euler / Heun / Euler ancestral RF / DPM++ 2M / DPM++ 2M SDE / ER-SDE**를 비교합니다. LoRA는 선택해서 적용할 수 있습니다.

1. 실행 환경·소스 준비 → 비교 조건 → LoRA → 텍스트 조건 → 샘플링 → 이미지 복원 → 비교표·모아보기 순으로 실행합니다.
2. 기본 실험은 같은 seed·30스텝으로 6장을 생성합니다. `MODE=similar_evaluations`는 예측 호출 수를 비슷하게 맞춥니다. Heun 호출 수는 `2 × steps − 1`이므로 일부 목표에서는 한 번 차이가 납니다.
3. `SEEDS`와 `TARGETS`에 쉼표로 여러 값을 넣으면 비교를 확장할 수 있습니다. 설정 셀에 전체 생성 수와 예측 호출 수가 표시됩니다.
4. 모델·LoRA·프롬프트·CFG·shift·크기·정밀도는 모든 비교에서 고정합니다. 확률적 방식은 추가 노이즈도 고정 seed로 재현합니다. SDE 두 방식의 시작 sigma 보정은 기존 구현을 따릅니다.
5. 시간은 CUDA 동기화 후 **샘플링 구간**을 측정합니다. 모델 로딩·텍스트 처리·VAE 복원은 제외하며, 단일 실행 시간은 근사 비교입니다. 속도 비교 시 Colab의 같은 GPU를 사용하세요.

GitHub 모드는 `feature/anima-samplers`가 원격에 올라온 뒤 실행할 수 있습니다. 현재 로컬 작업 폴더를 쓰려면 2번 셀에서 `SOURCE=local`을 선택하고 `LOCAL_REPO`를 지정합니다. GPU 런타임이 필요합니다.

In [ ]:
#@title 1. 생성 의존성 준비
import importlib.util, subprocess, sys

if importlib.util.find_spec('uv') is None:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'uv'], check=True)
uv_command = [sys.executable, '-m', 'uv', 'pip', 'install', '--python', sys.executable]
if sys.prefix == sys.base_prefix:
    uv_command.append('--system')
subprocess.run(uv_command + ['safetensors', 'einops', 'sentencepiece', 'tqdm',
                             'torchsde==0.2.6'], check=True)


In [ ]:
#@title 2. 샘플링 브랜치 또는 로컬 작업 폴더 준비
from pathlib import Path
import importlib, json, subprocess, sys, torch

SOURCE = 'github' #@param ['github', 'local']
REPO_URL = 'https://github.com/HisameOgasahara/tmp_diffsynth_practive.git' #@param {type:'string'}
REPO_REVISION = 'feature/anima-samplers' #@param {type:'string'}
LOCAL_REPO = '/content/tmp_diffsynth_practive' #@param {type:'string'}
GITHUB_CHECKOUT = '/content/tmp_diffsynth_sampler_compare' #@param {type:'string'}

if SOURCE == 'local':
    ROOT = Path(LOCAL_REPO).expanduser().resolve()
else:
    ROOT = Path(GITHUB_CHECKOUT).expanduser().resolve()
    if not ROOT.exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REVISION,
                        REPO_URL, str(ROOT)], check=True)
    else:
        current_branch = subprocess.check_output(
            ['git', '-C', str(ROOT), 'branch', '--show-current'], text=True).strip()
        if current_branch != REPO_REVISION:
            raise RuntimeError('기존 checkout의 브랜치가 다릅니다. 새 GITHUB_CHECKOUT 경로를 지정하세요.')
        print('기존 checkout을 사용합니다. 최신 코드는 새 경로에 clone하세요.')
if not (ROOT / 'src/anima_core/sampling').is_dir():
    raise RuntimeError('샘플링 코드가 있는 브랜치/로컬 폴더를 지정하세요.')
if not torch.cuda.is_available():
    raise RuntimeError('Colab 설정에서 GPU 런타임을 선택하세요.')
if 'anima_core.model_cache' in sys.modules:
    sys.modules['anima_core.model_cache'].clear_model_cache()
for name in list(sys.modules):
    if name == 'anima_core' or name.startswith('anima_core.'):
        del sys.modules[name]
sys.path.insert(0, str(ROOT / 'src'))
importlib.invalidate_caches()
from anima_core.sampling import SAMPLERS
from anima_core.runtime import configure_diagnostics
configure_diagnostics({'logging': {'level':'WARNING', 'progress':True},
                       'use_model_cache':False, 'profiler':{'enabled':False}})
SOURCE_INFO = {'root':str(ROOT), 'commit':subprocess.check_output(
    ['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip(),
    'working_tree':subprocess.check_output(
        ['git', '-C', str(ROOT), 'status', '--short'], text=True).strip()}
print('소스:', SOURCE_INFO)
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name())


## 비교 조건

`assets/examples/anima_minimal_config.json`의 프롬프트·negative prompt·생성 설정을 기본으로 사용합니다. `PROMPT_OVERRIDE`와 `NEGATIVE_OVERRIDE`가 비어 있으면 원래 프롬프트를 유지합니다. `CONFIG_JSON`에 기존 생성 설정 JSON을 지정할 수도 있습니다.

`MODE=same_steps`: 같은 스텝 수. `MODE=similar_evaluations`: 비슷한 예측 호출 수. `MODE=both`: 두 실험을 모두 수행합니다. 예측 호출 한 번은 CFG=1에서 DiT 한 번, 그 외에는 두 번입니다. Heun의 스텝 수가 달라지는 실험은 schedule의 방문 지점도 달라집니다.

In [ ]:
#@title 3. 비교 설정과 실행 목록
from datetime import datetime
from itertools import product
from uuid import uuid4

CONFIG_JSON = '' #@param {type:'string'}
PROMPT_OVERRIDE = 'shirakawa yuina, @WFS, blue headgear, platinum blonde hair, yuina eyelashes, blue capelet, 1girl, solo, lupper body, upper body, solo, expressionless, looking at viewer, blue capelet, , white uniform shirt, white pleated skirt, white pantyhose, sitting, cafe, blurry, depth of field, sunlight, vibrant colors, wallpaper' #@param {type:'string'}
NEGATIVE_OVERRIDE = 'worst quality, low quality, score_1, score_2, score_3, blurry, jpeg artifacts' #@param {type:'string'}
SAMPLER_LIST = 'euler,heun,euler_ancestral,dpmpp_2m,dpmpp_2m_sde,er_sde' #@param {type:'string'}
MODE = 'same_steps' #@param ['same_steps', 'similar_evaluations', 'both']
SEEDS = '' #@param {type:'string'}
TARGETS = '30' #@param {type:'string'}
CFG = 4.0 #@param {type:'number'}
WIDTH = 1216 #@param {type:'integer'}
HEIGHT = 832 #@param {type:'integer'}
SHIFT = 3.0 #@param {type:'number'}
DENOISE = 1.0 #@param {type:'number'}
DTYPE = 'float16' #@param ['float16', 'bfloat16', 'float32']
USE_TOKEN_WEIGHTS = True #@param {type:'boolean'}
ETA = 1.0 #@param {type:'number'}
S_NOISE = 1.0 #@param {type:'number'}
WARMUP = True #@param {type:'boolean'}
OUTPUT_ROOT = '/content/anima_sampler_comparison' #@param {type:'string'}

config_path = Path(CONFIG_JSON).expanduser() if CONFIG_JSON.strip() else ROOT / 'assets/examples/anima_minimal_config.json'
baseline = json.loads(config_path.read_text(encoding='utf-8'))
config = dict(baseline)
config.update(prompt=PROMPT_OVERRIDE or baseline['prompt'],
              negative_prompt=NEGATIVE_OVERRIDE or baseline['negative_prompt'],
              cfg=float(CFG), width=int(WIDTH), height=int(HEIGHT), shift=float(SHIFT),
              denoise=float(DENOISE), dtype=DTYPE, device='cuda',
              use_token_weights=bool(USE_TOKEN_WEIGHTS),
              lora_path='', lora_scale=1.0, random_seed=False,
              use_model_cache=False, profiler={'enabled':False},
              logging={'level':'WARNING','progress':True})
samplers = [name.strip() for name in SAMPLER_LIST.split(',') if name.strip()]
seeds = [int(value.strip()) for value in SEEDS.split(',') if value.strip()] or [int(baseline['seed'])]
targets = [int(value.strip()) for value in TARGETS.split(',') if value.strip()]
if not samplers or not targets or any(name not in SAMPLERS for name in samplers):
    raise ValueError('SAMPLER_LIST와 TARGETS를 확인하세요. 사용 가능: ' + ', '.join(SAMPLERS))
if any(value < 1 for value in targets):
    raise ValueError('TARGETS는 1 이상의 정수 목록이어야 합니다.')
if WIDTH < 16 or HEIGHT < 16 or WIDTH % 16 or HEIGHT % 16:
    raise ValueError('WIDTH와 HEIGHT는 16의 배수로 지정하세요.')
if not 0 <= ETA <= 1 or not 0 <= S_NOISE or not 0 < DENOISE <= 1 or not SHIFT > 0:
    raise ValueError('ETA 0~1, S_NOISE 0 이상, DENOISE 0~1(0 제외), SHIFT 양수로 지정하세요.')
if len(samplers) != len(set(samplers)) or len(seeds) != len(set(seeds)) or len(targets) != len(set(targets)):
    raise ValueError('샘플러·seed·TARGETS의 중복 값을 제거하세요.')
dtype = getattr(torch, config['dtype'])
device = config['device']
modes = ['same_steps', 'similar_evaluations'] if MODE == 'both' else [MODE]
cases = []
for mode, seed, target, sampler in product(modes, seeds, targets, samplers):
    steps = max(1, (target + 1) // 2) if mode == 'similar_evaluations' and sampler == 'heun' else target
    evaluations = 2 * steps - 1 if sampler == 'heun' else steps
    case_id = f'{mode}_seed-{seed}_target-{target}_{sampler}'
    cases.append({'id':case_id, 'mode':mode, 'seed':seed, 'target':target,
                  'sampler':sampler, 'steps':steps, 'expected_evaluations':evaluations})
RUN_DIR = Path(OUTPUT_ROOT).expanduser().resolve() / (datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid4().hex[:8])
RUN_DIR.mkdir(parents=True)
print('전체 생성 수:', len(cases), '| 예측 호출 합계:', sum(c['expected_evaluations'] for c in cases))
print('출력 폴더:', RUN_DIR)
print(json.dumps(config, ensure_ascii=False, indent=2))
for case in cases:
    print(case['mode'], '| seed', case['seed'], '| target', case['target'], '|', case['sampler'],
          '| steps', case['steps'], '| 예측 호출', case['expected_evaluations'])


In [ ]:
#@title 4. 선택: LoRA 준비
ENABLE_LORA = False #@param {type:'boolean'}
LORA_SOURCE = 'local' #@param ['local', 'google_drive', 'hugging_face']
LORA_PATH = '' #@param {type:'string'}
LORA_SCALE = 1.0 #@param {type:'number'}
HF_LORA_REPO = '' #@param {type:'string'}
HF_LORA_FILENAME = '' #@param {type:'string'}
HF_LORA_REVISION = 'main' #@param {type:'string'}

config['lora_path'] = ''
config['lora_scale'] = float(LORA_SCALE)
if ENABLE_LORA:
    if LORA_SOURCE == 'google_drive':
        from google.colab import drive
        drive.mount('/content/drive')
    elif LORA_SOURCE == 'hugging_face':
        from huggingface_hub import hf_hub_download
        if not HF_LORA_REPO.strip() or not HF_LORA_FILENAME.strip():
            raise ValueError('HF_LORA_REPO와 HF_LORA_FILENAME을 지정하세요.')
        LORA_PATH = hf_hub_download(repo_id=HF_LORA_REPO, filename=HF_LORA_FILENAME,
                                  revision=HF_LORA_REVISION)
    path = Path(LORA_PATH).expanduser().resolve() if LORA_PATH.strip() else None
    if path is None or not path.is_file():
        raise ValueError('사용할 Anima LoRA 파일 경로를 지정하세요.')
    config['lora_path'] = str(path)
print('LoRA:', config['lora_path'] or '사용 안 함', '| 강도:', config['lora_scale'])


In [ ]:
#@title 5. 모델 가중치와 공통 텍스트 조건 준비
import gc, platform, importlib.metadata
from anima_core.runtime import download_weights, load_tokenizers, encode_prompt
from anima_core.model_cache import use_text_encoder

configure_diagnostics(config)
weights = download_weights()
qwen, t5 = load_tokenizers()
raw = {}
with use_text_encoder(weights['text_encoder'], device, dtype, enabled=False) as text_encoder:
    for label, prompt in [('positive', config['prompt']), ('negative', config['negative_prompt'])]:
        embeds, ids, token_weights = encode_prompt(
            text_encoder, qwen, t5, prompt, device, dtype,
            use_token_weights=config['use_token_weights'], return_token_weights=True)
        raw[label + '_raw'] = embeds.cpu()
        raw[label + '_t5'] = ids.cpu()
        raw[label + '_weights'] = token_weights.cpu()
        del embeds, ids, token_weights
del text_encoder, qwen, t5
gc.collect()
torch.cuda.empty_cache()
torch.save(raw, RUN_DIR / 'conditioning.pt')
manifest = {'config':config.copy(), 'cases':cases, 'source':SOURCE_INFO,
            'eta':float(ETA), 's_noise':float(S_NOISE), 'warmup':bool(WARMUP),
            'baseline_config':str(config_path),
            'environment':{'python':platform.python_version(), 'torch':torch.__version__,
                           'cuda':torch.version.cuda, 'gpu':torch.cuda.get_device_name(),
                           'torchsde':importlib.metadata.version('torchsde')},
            'weights':{name:str(path) for name,path in weights.items()}}
(RUN_DIR / 'experiment.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('공통 텍스트 조건 준비 완료:', RUN_DIR / 'conditioning.pt')


## 샘플링과 이미지 복원

DiT와 LoRA를 한 번 로드하고 모든 경우를 순서대로 생성합니다. 각 latent는 CPU 파일로 저장한 뒤 GPU에서 해제합니다. 이후 DiT를 해제하고 VAE를 한 번 로드해 PNG로 복원합니다. 중단 전에 완료된 latent와 측정 결과는 출력 폴더에 남습니다.

6번 셀만 다시 실행하면 같은 비교 조건으로 재생성하여 해당 실행 폴더의 결과를 덮어씁니다. 프롬프트나 LoRA를 바꿀 때는 3번부터 다시 실행해 새 실행 폴더를 만듭니다. LoRA를 유지하려면 4번 셀도 다시 실행합니다.

In [ ]:
#@title 6. 공통 DiT·LoRA로 샘플러 비교
import csv, time
from anima_core.runtime import adapt_conditioning, sample_latents
from anima_core.model_cache import use_dit

raw = torch.load(RUN_DIR / 'conditioning.pt', map_location='cpu', weights_only=True)
measurements = []

def save_measurements():
    if not measurements:
        return
    with (RUN_DIR / 'measurements.csv').open('w', newline='', encoding='utf-8-sig') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(measurements[0]))
        writer.writeheader()
        writer.writerows(measurements)

with use_dit(weights['dit'], device, dtype, enabled=False,
             lora_path=config['lora_path'] or None, lora_scale=config['lora_scale']) as dit:
    positive = adapt_conditioning(dit, raw['positive_raw'].to(device=device, dtype=dtype),
                                 raw['positive_t5'].to(device),
                                 t5_weights=raw['positive_weights'].to(device=device, dtype=dtype))
    negative = adapt_conditioning(dit, raw['negative_raw'].to(device=device, dtype=dtype),
                                 raw['negative_t5'].to(device),
                                 t5_weights=raw['negative_weights'].to(device=device, dtype=dtype))
    if WARMUP:
        warmup_latent = sample_latents(dit, positive, negative, config['height'], config['width'],
                                     cases[0]['seed'], 1, config['cfg'], sampler='euler',
                                     denoise=config['denoise'], shift=config['shift'],
                                     device=device, dtype=dtype)
        torch.cuda.synchronize()
        del warmup_latent
    for case in cases:
        calls = [0]
        def count_calls(module, inputs, output):
            calls[0] += 1
        hook = dit.register_forward_hook(count_calls)
        try:
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
            started = time.perf_counter()
            latent = sample_latents(dit, positive, negative, config['height'], config['width'],
                                    case['seed'], case['steps'], config['cfg'],
                                    sampler=case['sampler'], eta=float(ETA), s_noise=float(S_NOISE),
                                    denoise=config['denoise'], shift=config['shift'],
                                    device=device, dtype=dtype)
            torch.cuda.synchronize()
            seconds = time.perf_counter() - started
            peak_gib = torch.cuda.max_memory_allocated() / 1024**3
        finally:
            hook.remove()
        prediction_calls = calls[0] // (1 if config['cfg'] == 1.0 else 2)
        if prediction_calls != case['expected_evaluations']:
            raise RuntimeError('예측 호출 수가 실험 계획과 일치하지 않습니다.')
        latent_path = RUN_DIR / (case['id'] + '.pt')
        torch.save(latent.cpu(), latent_path)
        row = {**case, 'prediction_calls':prediction_calls, 'dit_calls':calls[0],
               'sampling_seconds':round(seconds, 4), 'peak_gpu_gib':round(peak_gib, 3),
               'latent':latent_path.name, 'png':case['id'] + '.png'}
        measurements.append(row)
        save_measurements()
        del latent
        print(case['id'], '| 초:', row['sampling_seconds'], '| DiT:', calls[0])
del dit, positive, negative, raw
gc.collect()
torch.cuda.empty_cache()
print('샘플링 완료:', RUN_DIR / 'measurements.csv')


In [ ]:
#@title 7. VAE 복원과 원본 PNG 저장
from PIL.PngImagePlugin import PngInfo
from anima_core.runtime import decode_image
from anima_core.model_cache import use_vae

with (RUN_DIR / 'measurements.csv').open(encoding='utf-8-sig', newline='') as handle:
    measurements = list(csv.DictReader(handle))
with use_vae(weights['vae'], device, dtype, enabled=False) as vae:
    for row in measurements:
        latent = torch.load(RUN_DIR / row['latent'], map_location='cpu', weights_only=True).to(device=device, dtype=dtype)
        image = decode_image(vae, latent, device)
        metadata = PngInfo()
        metadata.add_text('experiment', json.dumps({'config':config, 'case':row,
                          'eta':ETA, 's_noise':S_NOISE}, ensure_ascii=False))
        image.save(RUN_DIR / row['png'], pnginfo=metadata)
        del latent, image
        print('저장:', row['png'])
del vae
gc.collect()
torch.cuda.empty_cache()


In [ ]:
#@title 8. 비교표와 샘플러별 이미지 모아보기
from collections import defaultdict
from html import escape
from PIL import Image, ImageDraw
from IPython.display import display, HTML

PREVIEW_WIDTH = 384 #@param {type:'integer'}
COLUMNS = 3 #@param {type:'integer'}
if PREVIEW_WIDTH < 64 or COLUMNS < 1:
    raise ValueError('PREVIEW_WIDTH는 64 이상, COLUMNS는 1 이상으로 지정하세요.')
with (RUN_DIR / 'measurements.csv').open(encoding='utf-8-sig', newline='') as handle:
    measurements = list(csv.DictReader(handle))
columns = [('mode','비교 방식'), ('seed','seed'), ('target','목표'), ('sampler','샘플러'),
           ('steps','스텝'), ('prediction_calls','예측 호출'), ('dit_calls','DiT 호출'),
           ('sampling_seconds','샘플링(초)'), ('peak_gpu_gib','최대 GPU 메모리(GiB)')]
header = ''.join('<th>' + escape(label) + '</th>' for _, label in columns)
rows = ['<tr>' + ''.join('<td>' + escape(str(row[key])) + '</td>' for key, _ in columns) + '</tr>'
        for row in measurements]
display(HTML('<table><thead><tr>' + header + '</tr></thead><tbody>' + ''.join(rows) + '</tbody></table>'))
groups = defaultdict(list)
for row in measurements:
    groups[(row['mode'], row['seed'], row['target'])].append(row)
for (mode, seed, target), group in groups.items():
    thumb_height = round(PREVIEW_WIDTH * config['height'] / config['width'])
    tile_height = thumb_height + 48
    width = PREVIEW_WIDTH * min(COLUMNS, len(group))
    sheet = Image.new('RGB', (width, tile_height * ((len(group) + COLUMNS - 1) // COLUMNS)), 'white')
    draw = ImageDraw.Draw(sheet)
    for index, row in enumerate(group):
        x, y = (index % COLUMNS) * PREVIEW_WIDTH, (index // COLUMNS) * tile_height
        draw.text((x + 8, y + 6), row['sampler'], fill='black')
        label = f"steps={row['steps']} NFE={row['prediction_calls']} time={row['sampling_seconds']}s"
        draw.text((x + 8, y + 23), label, fill='black')
        with Image.open(RUN_DIR / row['png']) as original:
            thumbnail = original.convert('RGB')
        thumbnail.thumbnail((PREVIEW_WIDTH, thumb_height), Image.Resampling.LANCZOS)
        sheet.paste(thumbnail, (x, y + 48))
    sheet_path = RUN_DIR / f'comparison_{mode}_seed-{seed}_target-{target}.png'
    sheet.save(sheet_path)
    print('비교:', mode, '| seed:', seed, '| 목표:', target)
    display(sheet)
    print(sheet_path)
print('원본 PNG·latent·설정·측정 CSV:', RUN_DIR)


In [ ]:
#@title 9. 선택: 실험 결과 ZIP 보관
import shutil

MAKE_ZIP = False #@param {type:'boolean'}
DOWNLOAD_ZIP = False #@param {type:'boolean'}
if MAKE_ZIP or DOWNLOAD_ZIP:
    archive = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=str(RUN_DIR))
    print('ZIP:', archive)
    if DOWNLOAD_ZIP:
        from google.colab import files
        files.download(archive)


## 결과 읽기와 다음 실험

- 얼굴·손·의상 디테일과 프롬프트 반영은 원본 PNG로 확인합니다. 모아보기 이미지는 축소본입니다.
- 처음에는 같은 seed·30스텝 6종을 비교한 뒤, `TARGETS=10,20,30` 또는 추가 seed로 결과의 일관성을 살펴볼 수 있습니다.
- `MODE=both`를 선택하면 같은 스텝 수와 비슷한 예측 호출 수를 함께 볼 수 있습니다. 예측 호출 수가 같아도 갱신식·노이즈 비용 때문에 생성 시간은 다를 수 있습니다.
- LoRA 영향을 비교하려면 동일 조건으로 별도 실행 폴더를 만들고 `ENABLE_LORA`만 바꿉니다. 기본 프롬프트에는 기존 캐릭터 태그가 포함되어 있습니다.
- `experiment.json`에 전체 조건·소스 상태·환경, `measurements.csv`에 시간·호출 수·GPU 메모리, PNG 메타데이터에 개별 생성 조건을 기록합니다.
- 원격에서 이 노트북을 다시 실행하려면 노트북과 샘플링 코드가 같은 revision에 있어야 합니다. Colab 종료 전에 결과 ZIP을 보관하세요.